# آموزشِ صدا — نسخهٔ Kaggle

**فرقش با Colab:** اینجا اجرا روی سرورِ Kaggle در پس‌زمینه می‌رود.
مرورگر را می‌بندید، کامپیوتر را خاموش می‌کنید، اینترنتتان قطع می‌شود —
هیچ‌کدام اثری ندارد.

## یک بار: ضبط‌ها را به‌صورتِ Dataset بگذارید

دو اجرا دقیقاً سرِ دانلودِ اولین فایل از گوگل‌درایو کشته شد. پس دیگر
از درایو نمی‌گیریمشان؛ روی خودِ Kaggle می‌گذاریمشان و هر اجرا فقط
می‌خوانَدشان — بی دانلود، بی اشتراک‌گذاری، و بی چیزی که بشکند.

۱. بالا سمتِ چپ: `Datasets` ← `New Dataset`
۲. چهار فایلِ صوتیِ رضوی را بکشید داخل، اسمش را بگذارید `razavi-voice`،
   `Create` (خصوصی بماند؛ لازم نیست عمومی باشد)
۳. برگردید به همین نوت‌بوک ← پنلِ راست ← `+ Add Input` ← `Datasets` ←
   `razavi-voice` ← `+`

این کار فقط یک بار است. اجراهای بعدی همان دیتاست را دارند.

## هر اجرا

۱. پنلِ راست ← `Session options`
   - **Accelerator** ← `GPU T4 x2`
   - **Internet** ← `On`
     (برای وزن‌های پایه لازم است. اگر خاموش بود، حساب باید با شمارهٔ
     موبایل تأیید شده باشد: `Settings` ← `Phone Verification`)
۲. دکمهٔ **`Save Version`** بالا سمتِ راست ←
   **`Save & Run All (Commit)`** ← `Save`

**یک بار بزنید، نه دو بار.** اگر وقتی نسخه‌ای در حالِ اجراست دوباره
`Save` بزنید، قبلی لغو می‌شود. از منوی سه‌نقطه ← `Versions` می‌شود
دید چند نسخه در فهرست است.

حالا می‌توانید تب را ببندید.

## بعدش

یکی-دو ساعت بعد به همان صفحه برگردید و از تبِ `Output` دو فایل را
دانلود کنید. بعد در درایو، پوشهٔ `voice-models` بگذاریدشان.

## اگر «Exit code: 137» گرفت

۱۳۷ یعنی سیستم فرایند را کُشت (`SIGKILL`) و از بیرونِ فرایند هیچ
دلیلی در آن نیست. پس این نوت‌بوک بعدِ هر قدمِ سنگین رم و دیسک را چاپ
می‌کند و هیچ قدمی خاموش نیست: **آخرین خطِ لاگ** می‌گوید کجا بوده و
حافظه تنگ بوده یا نه.

اجرای دوم دقیقاً همین را روشن کرد — رم ۱٫۳ گیگ از ۳۱، دیسک ۱۹ گیگ
آزاد، و مرگ سرِ اولین دانلود. حافظه نبود؛ دانلود بود. برای همین
دانلود حذف شد.

## اگر نصفه ماند و باید ادامه بدهید

خروجیِ هر اجرای کامل یک پوشهٔ `resume/` هم دارد. در اجرای بعدی
`+ Add Input` ← `Your Work` ← همین نوت‌بوک را اضافه کنید؛ آموزش از
همان‌جا ادامه می‌دهد و از صفر شروع نمی‌کند.

---

منطقِ قدم‌ها در مخزن است و همین نوت‌بوک از آنجا می‌گیردش — همان کدی
که نسخهٔ Colab هم اجرا می‌کند.


## ۱ — کارتِ گرافیک، اینترنت، و جا

هر سه باید باشند، وگرنه بقیه بی‌معنی است.


In [ ]:
import subprocess, sys, os, shutil, json, time, glob

T0 = time.time()

# ══ چرا خطِ به خط ══
# وقتی خروجی به فایل می‌رود (و در Kaggle می‌رود)، پایتون آن را بلوکی
# نگه می‌دارد. یعنی اگر فرایند با SIGKILL بمیرد، هرچه در بافر مانده
# **هیچ‌وقت نوشته نمی‌شود** — و لاگ روی آخرین خطی می‌ایستد که اتفاقاً
# بافر را پر کرده بود، نه آخرین کاری که واقعاً انجام شده. اجرای اولِ
# Kaggle دقیقاً همین شکل بود: لاگ روی «وزن‌ها آماده‌اند» تمام شده بود
# و بعدش هیچ. این یک خط یعنی هر خط همان لحظه روی دیسک است.
try:
    sys.stdout.reconfigure(line_buffering=True)
except (AttributeError, ValueError):
    pass


def memNow_():
    """حافظهٔ مصرفی و سقفش — از cgroup، نه از /proc/meminfo.

    داخلِ کانتینر `/proc/meminfo` حافظهٔ **میزبان** را می‌گوید، نه سهمِ
    این کانتینر. یعنی می‌شود «۱٫۳ از ۳۱٫۳ گیگ» چاپ کرد و یک قدم بعد
    OOM گرفت. عددی که به آن استناد می‌کنیم باید همان عددی باشد که
    کُشنده به آن نگاه می‌کند، وگرنه ابزارِ سنجش خودش دروغ می‌گوید.
    """
    pairs = (('/sys/fs/cgroup/memory.current', '/sys/fs/cgroup/memory.max'),
             ('/sys/fs/cgroup/memory/memory.usage_in_bytes',
              '/sys/fs/cgroup/memory/memory.limit_in_bytes'))
    for cur, mx in pairs:
        try:
            u = int(open(cur).read().strip())
            t = open(mx).read().strip()
            t = None if t == 'max' else int(t)
        except (IOError, OSError, ValueError):
            continue
        if t and t < (1 << 50):          # سقفِ نجومی یعنی «بی‌سقف»
            return u / float(1 << 30), t / float(1 << 30), 'cgroup'
    m = {}
    for ln in open('/proc/meminfo'):
        q = ln.split()
        if len(q) > 1:
            m[q[0].rstrip(':')] = int(q[1]) / 1048576.0
    return (m.get('MemTotal', 0.0) - m.get('MemAvailable', 0.0),
            m.get('MemTotal', 0.0), 'میزبان')


def res(tag=''):
    """رم و دیسکِ همین لحظه — بعدِ هر قدمِ سنگین.

    اجرای اولِ Kaggle با «Canceled by backend. Exit code: 137» مُرد و
    لاگ چیزی نداشت: ۱۳۷ یعنی SIGKILL، و کُشنده هیچ‌وقت خودش را معرفی
    نمی‌کند. تنها راهِ فهمیدنش این است که پیش از مرگ خودمان عدد
    گذاشته باشیم — همان درسِ «گزارشی که خوانده نمی‌شود».

    و اجرای دوم همین را ثابت کرد: رم ۱٫۳ گیگ بود و دیسک ۱۹ گیگ آزاد،
    و مرگ سرِ اولین دانلود. یک اندازه‌گیری که یک فرضیه را **رد** کند
    همان‌قدر ارزش دارد که یکی که تأییدش کند.
    """
    u, t, src = memNow_()
    where = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/'
    d = shutil.disk_usage(where)
    print('   [%5.0f ثانیه]  رم %.1f از %.1f گیگ (%s)  ·  دیسکِ آزاد %.0f گیگ   %s'
          % (time.time() - T0, u, t, src, d.free / float(1 << 30), tag),
          flush=True)


g = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True)
name = (g.stdout or b'').decode().strip()
if g.returncode != 0 or not name:
    raise SystemExit('GPU روشن نیست: Session options ← Accelerator ← GPU T4')
print('کارت:', name.splitlines()[0])

# اینترنت را همین‌جا می‌سنجیم، نه سه سلول بعد وقتی دانلود می‌شکند.
r = subprocess.run(['curl', '-sSfI', '-m', '20',
                    'https://raw.githubusercontent.com'],
                   capture_output=True)
if r.returncode:
    raise SystemExit('اینترنت خاموش است: Session options ← Internet ← On')
print('اینترنت: وصل')
res('آغاز')

## ۲ — تنظیمات


In [ ]:
VOICE = 'razavi'

SR      = '40k'
EPOCHS  = 150
BATCH   = 8
SAVE_EVERY = 25

# در Kaggle خروجی از این پوشه برداشته می‌شود.
OUT  = '/kaggle/working'
WORK = OUT + '/work'

# پوشه‌ای که اجرای بعدی از آن ادامه می‌دهد. اگر خروجیِ یک اجرای قبلی
# را به‌عنوانِ ورودی اضافه کرده باشید، همین‌جا پیدا می‌شود.
RESUME = OUT + '/resume'

# ══ ضبط‌ها از Dataset می‌آیند، نه از اینترنت ══
# اینجا فقط می‌گوییم چه چیزی صوت حساب می‌شود. ولی **نامشان** باید
# جایی بماند: وقتی شناسه‌های درایو از این سلول برداشته شد، تنها جایی
# که ثبت بود کدام چهار فایل آموزش می‌دهند هم رفت. یک مسیرِ اجرا که
# حذف می‌شود نباید شاهدِ تاریخی‌اش را هم با خودش ببرد.
#
# دیتاستِ `razavi-voice` از این چهار تا ساخته شده — پوشهٔ «TEST» در
# درایو (10hj4y9DfT52kY0_XEqVDKFwDI_jhSM1c):
#   D1736591T18498032(Web).mp3   ۷٫۴ مگابایت
#   D1736592T10213625(Web).mp3   ۶٫۷ مگابایت
#   D1736591T18630070(Web).mp3   ۶٫۵ مگابایت
#   D1738956T14935309(Web).mp3   ۵٫۳ مگابایت
# روی هم ۳۹ دقیقه و ۲۴ ثانیه صدای تمیز، بعدِ سه دروازه.
AUDIO_EXT = ('.mp3', '.wav', '.m4a', '.ogg', '.flac', '.aac', '.opus', '.wma')

## ۳ — کد


In [ ]:
os.chdir(OUT)
if not os.path.isdir(OUT + '/rvc'):
    subprocess.run(['git', 'clone', '--depth', '1', '-q',
                    'https://github.com/RVC-Project/'
                    'Retrieval-based-Voice-Conversion-WebUI',
                    OUT + '/rvc'], check=True)

RAW = 'https://raw.githubusercontent.com/mahdighandi1989/Content-Engine/main/tools'
for f in ('rvcpipe.py', 'dsprep.py'):
    subprocess.run(['curl', '-sSLf', RAW + '/' + f,
                    '-o', OUT + '/' + f], check=True)
sys.path.insert(0, OUT)
import rvcpipe as P, dsprep as D

ROOT = OUT + '/rvc'
os.makedirs(WORK, exist_ok=True)
print('قدم‌ها:', [n for n, _ in P.steps(VOICE, '/x', ROOT, sr=SR)])
res('کد')

## ۴ — وابستگی‌ها


In [ ]:
deps = P.TRAIN_DEPS + D.DS_DEPS
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q']
               + deps, check=True)
print('نصب شد:', len(deps), 'بسته')
res('وابستگی‌ها')

## ۵ — وزن‌های پایه

حدود ۵۵۰ مگابایت. پروانه‌ها سنجیده شده: کدِ RVC ‏MIT · وزن‌ها ‏MIT ·
ContentVec ‏MIT · RMVPE ‏Apache-2.0 · مدلِ گوینده ‏Apache-2.0.


In [ ]:
os.chdir(ROOT)
hf = shutil.which('hf') or shutil.which('huggingface-cli')
for cmd in P.assetCmds_(py=sys.executable, sr=SR, hf=hf or 'hf'):
    if cmd[:3] == [sys.executable, '-m', 'pip']:
        continue
    if cmd[0] in ('hf', 'huggingface-cli'):
        cmd[0] = hf or cmd[0]
    r = subprocess.run(cmd)
    if r.returncode:
        raise SystemExit('دانلود ناموفق: ' + ' '.join(cmd[:4]))
print('وزن‌ها آماده‌اند')
res('وزن‌ها')

## ۶ — ضبط‌ها

از پوشهٔ `input` برداشته می‌شوند، نه از اینترنت. هیچ دانلودی در کار
نیست، پس هیچ‌چیز هم نمی‌تواند سرِ دانلود بشکند.

In [ ]:
srcs = []
for base, _dirs, fs in os.walk('/kaggle/input'):
    # ══ خروجیِ خودمان ورودیِ خودمان نیست ══
    # اگر خروجیِ یک اجرای پیشین را برای «ادامه» اضافه کرده باشید،
    # نمونه‌های داوریِ همان اجرا (SAMPLE-*.wav) هم داخلش هستند. اگر
    # به‌عنوانِ ضبطِ خام برداشته شوند، مدل روی خروجیِ خودش آموزش
    # می‌بیند — خطایی که هیچ پیامی ندارد.
    if os.path.isdir(os.path.join(base, 'resume')):
        continue
    for f in sorted(fs):
        if f.lower().endswith(AUDIO_EXT) and not f.startswith('SAMPLE-'):
            srcs.append(os.path.join(base, f))
srcs.sort()
for s in srcs:
    print('  %6.0f مگابایت  %s'
          % (os.path.getsize(s) / 1048576.0, os.path.basename(s)), flush=True)
print('%d ضبط پیدا شد' % len(srcs))
res('ضبط‌ها')
if not srcs:
    raise SystemExit(
        'هیچ صوتی در /kaggle/input نیست. در پنلِ راست: '
        '+ Add Input ← Datasets ← دیتاستِ صداهای رضوی را اضافه کنید.')

## ۷ — جداکردنِ موسیقی و ساختِ دیتاست

سه دروازه: فاصله‌های بلند، کفِ هر تکه، و شباهت به گویندهٔ غالب.
روی چهار ضبطِ بلند چند دقیقه طول می‌کشد، پس هر فایل به‌محضِ تمام‌شدن
گزارش می‌دهد.


In [ ]:
DS = WORK + '/dataset'
os.makedirs(DS, exist_ok=True)
have = [f for f in os.listdir(DS) if f.endswith('.wav')]
if have:
    print('دیتاست از پیش آماده است: %d تکه' % len(have))
else:
    def onFile(rows):
        r_ = rows[-1]
        print('  %-22s %7.1f ثانیه → %3d تکه'
              % (r_['file'][:22], r_['seconds'], r_.get('segments', 0)),
              flush=True)
        res('تحلیلِ %d از %d' % (len(rows), len(srcs)))

    segs, rep = D.buildDataset_(srcs, DS, sampleDir=OUT, onFile=onFile)
    print('\n' + rep['line'])
    res('دیتاست')
    if not segs:
        raise SystemExit('هیچ تکه‌ای نماند — ضبط‌ها را ببینید')

## ۸ — ادامه از اجرای پیشین (اگر باشد)

اگر خروجیِ یک اجرای قبلی را به‌عنوانِ ورودی اضافه کرده باشید،
چک‌پوینت‌هایش برداشته می‌شود و آموزش از همان‌جا ادامه می‌دهد؛
`train.py` خودش آخرین چک‌پوینت را پیدا می‌کند.

استخراج دوباره اجرا می‌شود — چند دقیقه است و نتیجه‌اش قطعی است.
آنچه ارزشِ حمل دارد ساعت‌های آموزش است، نه آن.


In [ ]:
dst = os.path.join(ROOT, 'logs', VOICE)
os.makedirs(dst, exist_ok=True)
found = sorted(glob.glob('/kaggle/input/*/resume/' + VOICE + '/*.pth'))
for p in found:
    shutil.copy(p, dst)
    print('  برداشته شد: %s (%.0f مگابایت)'
          % (os.path.basename(p), os.path.getsize(p) / 1048576.0))
print('ادامه از %d چک‌پوینت' % len(found) if found
      else 'اجرای تازه — چک‌پوینتی از پیش نیست')

## ۹ — آموزش


In [ ]:
P.preLog_(ROOT, VOICE)
env = P.env(ROOT)
steps = P.steps(VOICE, DS, ROOT, sr=SR, f0method='rmvpe', epochs=EPOCHS,
                save_every=SAVE_EVERY, version='v2', gpus='0', n_p=2,
                batch=BATCH, py=sys.executable, latest=1)
done = os.path.join(ROOT, 'logs', VOICE, '3_feature768')
for nm, cmd in steps:
    if nm in ('preprocess', 'extract_f0', 'extract_feature') \
            and os.path.isdir(done) and os.listdir(done):
        print('%s: از پیش انجام شده' % nm)
        continue
    if nm == 'train':
        info = P.preTrain_(ROOT, VOICE, sr=SR, version='v2')
        print('فهرستِ آموزش:', info)
        if not info['from_dataset']:
            raise SystemExit('فهرست خالی است — استخراج چیزی نساخت')
    t0 = time.time()
    print('\n=== %s ===' % nm, flush=True)
    r = subprocess.run(cmd, cwd=ROOT, env=env)
    print('%s: %ds' % (nm, time.time() - t0))
    res(nm)
    if r.returncode:
        raise SystemExit('قدمِ «%s» شکست خورد (کد %d)' % (nm, r.returncode))

## ۱۰ — نتیجه

دو فایل در `/kaggle/working` می‌نشیند و در تبِ **Output** قابلِ دانلود
است. همین دو تا چیزی است که موتور به کار می‌برد.

کنارشان پوشهٔ `resume/` می‌ماند تا اجرای بعدی — اگر خواستید بیشتر
آموزش بدهید — از صفر شروع نکند. پوشهٔ کار پاک می‌شود تا خروجی سبک
بماند.


In [ ]:
o = P.outputs(VOICE, ROOT)
if not os.path.exists(o['model']):
    raise SystemExit('آموزش تمام شد ولی مدلی ساخته نشد: ' + o['model'])
saved = [shutil.copy(o['model'], OUT)]
for f in sorted(os.listdir(o['index_dir'])):
    if f.endswith('.index'):
        saved.append(shutil.copy(os.path.join(o['index_dir'], f), OUT))
for s in saved:
    print('%8.1f مگابایت  %s' % (os.path.getsize(s) / 1048576.0, s))

# ── توشهٔ اجرای بعدی ──
# فقط چک‌پوینت‌ها. ویژگی‌های استخراج‌شده چند گیگ‌اند و در چند دقیقه
# دوباره ساخته می‌شوند؛ آموزش نه.
keep = os.path.join(RESUME, VOICE)
os.makedirs(keep, exist_ok=True)
for f in sorted(glob.glob(os.path.join(ROOT, 'logs', VOICE, '*.pth'))):
    shutil.copy(f, keep)
    print('  توشه: %s (%.0f مگابایت)'
          % (os.path.basename(f), os.path.getsize(f) / 1048576.0))

shutil.rmtree(WORK, ignore_errors=True)
shutil.rmtree(ROOT, ignore_errors=True)
res('پایان')
print('\nتمام شد. از تبِ Output برشان دارید.')